# Parte 1: Preprocesamiento
## Tratamiento de Nulos y Datos corruptos

In [ ]:
import pandas as pd
import numpy as np

# Enlace de Google Drive para descargar el dataset completo:
# URL_DRIVE: https://drive.google.com/file/d/1oqtkbydpcCM_pWtkMoOpbd6uh65XnOiw/view?usp=sharing

PATH_DATASET = '../data/4_YouTubeDataset_withChannelElapsed.csv'

df = pd.read_csv(PATH_DATASET)

# Tratamiento de la variable likes/dislikes con eliminación de columnas
if 'likes/dislikes' in df.columns:
    df = df.drop(columns=['likes/dislikes'])

# Imputación por cero absoluto en ratios de interacción comunitaria (comments/views)
df['comments_per_view'] = df['comments_per_view'].fillna(0.0)

La variable de interacciones (likes/dislikes) contenía una corrupción estructural masiva ($96.50\%$ de valores centinela $-2$). Mantenerla mediante imputaciones artificiales habría inducido sesgos críticos y correlaciones mínimas en el modelo. Por otro lado, para los registros restantes sin comentarios, la imputación por cero absoluto ($0.0$) está metodológicamente justificada: en YouTube, la falta de comentarios representa de manera orgánica un estado real del negocio (un video que no logró activar a la comunidad), y no una pérdida aleatoria de información.

## Manejo de Outliers y Escalado

In [ ]:
# 1. Filtrado de inconsistencias físicas severas (0.02% del dataset)
df = df[(df['views'] >= 0) & (df['comments'] >= 0)]

# 2. Ingeniería de variables temporales para el Módulo Optimizador
df['videoPublished_dt'] = pd.to_datetime(df['videoPublished'], errors='coerce', utc=True)
df['publish_year'] = df['videoPublished_dt'].dt.year
df['publish_month'] = df['videoPublished_dt'].dt.month
df['publish_day_of_week'] = df['videoPublished_dt'].dt.dayofweek
df['publish_hour'] = df['videoPublished_dt'].dt.hour

# 3. Aislamiento de la Variable Objetivo y Características (Features)
y = df['views/elapsedtime']
columnas_X = [
    'channelId', 'videoCategoryId', 'channelViewCount', 'videoCount', 
    'subscriberCount', 'channelelapsedtime', 'publish_year', 
    'publish_month', 'publish_day_of_week', 'publish_hour'
]
X = df[columnas_X]

Las variables de rendimiento en redes sociales sufren de una asimetría positiva severa dictada por el fenómeno de la Cola Larga (Long Tail), donde los videos virales rompen cualquier escala de distribución normal. Eliminar estos outliers tradicionales destruiría el propósito comercial del simulador, el cual busca justamente modelar el éxito masivo. Por esto, la estrategia consistió en:  
* Filtrar inconsistencias físicas, eliminando del $0.02\%$ de filas con métricas negativas imposibles.  
* Se descartó MinMaxScaler porque la presencia de un solo video ultra-viral con miles de millones de vistas comprimiría al $99\%$ de los datos normales en un rango diminuto entre $0.00$ y $0.01$, destruyendo la varianza. StandardScaler, al centrar en la media y dividir por la desviación estándar, no posee un límite fijo, permitiendo que los canales promedio mantengan una separación matemática saludable mientras los videos virales se posicionan naturalmente a varias desviaciones estándar de distancia.

## Transformación de Columnas

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from category_encoders import TargetEncoder

# División metodológica estricta (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definición de familias de características
features_escalar = ['channelViewCount', 'videoCount', 'subscriberCount', 'channelelapsedtime',
                    'publish_year', 'publish_month', 'publish_day_of_week', 'publish_hour']

# Pipeline de preprocesamiento encapsulado
preprocesador = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), features_escalar),
        ('cat_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['videoCategoryId']),
        ('target_enc', TargetEncoder(smoothing=10.0), ['channelId'])
    ],
    remainder='drop'
)

Para codificar la variable de alta cardinalidad 'channelId' (más de miles de canales únicos), se aplicó Target Encoding. Esta técnica reemplaza el ID por el promedio de la velocidad de tracción (views/elapsedtime) histórica del canal, capturando de forma directa el "peso" y autoridad del creador de contenido.  
No se utilizaron herramientas de automatización como LazyPredict porque son incapaces de manejar pipelines complejos de manera interna. Al requerir que el Target Encoding se procese fuera de la partición de datos, introducen un sesgo masivo por Fuga de Datos (Data Leakage), provocando métricas falsas de éxito que colapsarían en producción. Además, su ineficiencia computacional congelaría el entorno al intentar correr modelos no paramétricos pesados (como SVR o Gaussian Process) sobre una matriz de más de $575,000$ registros.

# Parte 2: Selección de Modelos
## Comparación de modelos
| Modelo | R² | MAE | RMSE | Tiempo fit (s) | Tiempo pred (s) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| RandomForest(200) | 0.0375 | 2.4400 | 44.9589 | 183.73 | 0.420 |
| XGBoost | 0.0201 | 2.4254 | 45.3618 | 7.00 | 0.262 |
| LightGBM | 0.0068 | 2.4636 | 45.6704 | 3.41 | 0.293 |
| Lasso | -0.0082 | 2.6295 | 46.0143 | 3.75 | 0.232 |
| Ridge | -0.0093 | 2.6271 | 46.0393 | 3.47 | 0.235 |
| Lineal | -0.0093 | 2.6271 | 46.0393 | 6.21 | 0.438 |
| KNN(k=10) | -0.0141 | 1.8113 | 46.1475 | 4.05 | 2.494 |
| GradientBoosting | -0.0573 | 2.4445 | 47.1210 | 481.74 | 0.665 |
| Árbol(d=10) | -0.1619 | 2.5215 | 49.3969 | 6.39 | 0.146 |

Tras un screening inicial por división simple (Holdout), los modelos lineales y KNN demostraron un $R^2$ negativo y tiempos de inferencia prohibitivos para producción (KNN tardó $92.49$ segundos en predecir debido a que calcula distancias contra todo el dataset en tiempo real). Los tres primeros lugares están ocupados por algoritmos basados en árboles de ensamble (Random Forest, XGBoost y LightGBM). Son los únicos que lograron obtener un $R^2$ positivo. En consecuencia, se seleccionó el Top 3 de modelos basados en ensamble para una validación cruzada formal de 5 pliegues. 

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

# 1. Definimos los 3 modelos finalistas del screening anterior
finalistas = {
    'Random Forest (50)': make_pipeline(preprocesador, RandomForestRegressor(n_estimators=50, max_depth=12, n_jobs=-1, random_state=42)), # Bajamos estimadores para que vuele
    'XGBoost':             make_pipeline(preprocesador, XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)),
    'LightGBM':            make_pipeline(preprocesador, LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1))
}

print("======= VALIDACIÓN CRUZADA FORMAL (CV=5) =======")
for nombre, pipe_finalista in finalistas.items():
    # Calculamos el MAE usando cross_val_score (retorna valores negativos, aplicamos abs)
    scores_mae = cross_val_score(pipe_finalista, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_promedio = np.mean(np.abs(scores_mae))
    
    # Calculamos el R² en validación cruzada
    scores_r2 = cross_val_score(pipe_finalista, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    r2_promedio = np.mean(scores_r2)
    
    print(f"{nombre} en CV -> MAE Promedio: {mae_promedio:.4f} | R² Promedio: {r2_promedio:.4f}")
print("==================================================")

| Modelo | MAE Promedio | R² Promedio |
| :--- | :--- | :--- |
| Random Forest (50) | 2.3427 | -0.3071 |
| XGBoost | 2.3210 | -0.0739 |
| LightGBM | 2.3320 | 0.0147 |

Los datos expusieron un comportamiento contundente. En una división simple previa, Random Forest mostró un espejismo de éxito con un $R^2$ positivo, pero al someterlo a la estabilidad de la Validación Cruzada ($CV=5$), su rendimiento se desplomó a $-0.3071$ debido a un severo sobreajuste. LightGBM se consolidó como el único algoritmo capaz de mantener un $R^2$ promedio positivo (0.0147) en todas las particiones. Además, a nivel de infraestructura, LightGBM entrenó en solo $3.41$ segundos en contraste con los más de $183$ segundos de Random Forest. Esto garantiza viabilidad técnica para construir un Módulo Optimizador capaz de realizar múltiples simulaciones web en milisegundos.

# Parte 3: Optimización
## Selección de Métrica 
En este proyecto se determinó guiar la optimización apuntando a la maximización del MAE (Mean Absolute Error) sobre $R^2$ por razones estratégicas de negocio:  
* Interpretabilidad Directa: El $R^2$ es un porcentaje adimensional complejo de comunicar a un usuario final. El MAE entrega un valor actionable en la misma unidad que el objetivo: "El simulador se equivoca, en promedio, por X cantidad de vistas/tiempo".  
* Robustez ante Outliers Extremos: Al seguir una distribución de "Cola Larga", el $R^2$ y el MSE castigan con fuerza cuadrática los errores en videos ultra-virales, forzando al modelo a sesgarse hacia eventos astronómicos. El MAE trata cada error de forma equitativa, logrando un modelo estable para el $99\%$ de los creadores de contenido tradicionales.  
* Sensibilidad al Rango de Operación: Para un recomendador de escenarios temporales, es crítico que el modelo sea consistente en el rango normal de visualizaciones en lugar de perseguir la predicción exacta de anomalías virales.  

## GridSearchCV y Optuna
Para afinar el motor predictivo, se contrastó una búsqueda tradicional (GridSearchCV) contra una Optimización Bayesiana de última generación utilizando Optuna. Mientras que GridSearchCV se limitó a combinaciones discretas (reduciendo el MAE a $2.1281$), Optuna modeló probabilísticamente el espacio de búsqueda continuo para encontrar el "punto dulce" exacto de los parámetros en solo $160.1$ segundos.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Espacio de búsqueda continuo y condicional optimizado para Gradient Boosting
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 50, 300),
        'max_depth':       trial.suggest_int('max_depth', 3, 20),
        'num_leaves':      trial.suggest_int('num_leaves', 20, 100),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0)
    }
    
    modelo = LGBMRegressor(**params, random_state=42, n_jobs=-1, verbose=-1)
    pipe = Pipeline(steps=[('preprocesamiento', preprocesador), ('estimador', modelo)])
    
    # Maximizar el MAE negativo equivale a minimizar el error absoluto real
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

Hiperparámetros Óptimos Encontrados:
* n_estimators: 50

* max_depth: 17

* num_leaves: 20

* learning_rate: 0.04718

* subsample: 0.94649

Análisis de la Arquitectura Ganadora: El comportamiento hallado por Optuna es brillante: el algoritmo configuró un modelo profundo pero extremadamente estrecho (max_depth: 17, num_leaves: 20). Al limitar de forma nativa el crecimiento de las hojas independientes dentro de un árbol profundo, LightGBM captura interacciones altamente específicas y no lineales (ej. el cruce exacto de una categoría con un rango de hora) sin inflar la complejidad del ensamble. La tasa de aprendizaje pausada (0.047) y el muestreo del $94.6\%$ de los datos por árbol (subsample) actúan como robustas barreras defensivas contra el sobreajuste al ruido de la plataforma.

## Entrenamiento y Feature Importance
| Posición | Variable | Importancia por GAIN (Poder Predictivo) | Importancia por SPLIT (Frecuencia) |
| :---: | :--- | :---: | :---: |
| 1 | `channelId` | 610,381,664.0 | 102 |
| 2 | `videoCount` | 385,028,518.0 | 222 |
| 3 | `subscriberCount` | 124,061,355.0 | 98 |
| 4 | `channelViewCount` | 110,462,514.0 | 108 |
| 5 | `publish_hour` | 91,989,048.0 | 100 |

El análisis de importancia por GAIN valida científicamente la coherencia del modelo frente al negocio. Las variables de autoridad histórica del creador (channelId, videoCount, subscriberCount) controlan la mayor reducción del error, confirmando que la inercia del canal es el predictor dominante. Crucialmente, la publish_hour (Hora de publicación) se posiciona firmemente en el Top 5 de ganancia y con una alta frecuencia de uso (Split: 100). Esto provee el sustento matemático que viabiliza el Módulo Optimizador: el modelo detecta suficiente varianza en las horas de publicación como para ser capaz de recomendar escenarios óptimos de lanzamiento para maximizar vistas.

## Comparación antes y después de la Optimización

| Módulo del Proceso | MAE en Set de Test | R² en Set de Test | Estado y Progreso |
| :--- | :---: | :---: | :--- |
| **Modelo Inicial (Base)** | 2.4636 | 0.0147 | Desempeño por defecto inicial. |
| **GridSearchCV Exhaustivo** | 2.1281 | -0.0739 | Ajuste por grilla discreta rígida. |
| **Optuna (Optimización Bayesiana)** | 1.9727 | 0.0107 | **Ganador Definitivo (Reducción del ~20% del error)** |

Al aislar con éxito la interferencia de los videos virales mediante el uso adaptativo de StandardScaler, e implementar un preprocesamiento robusto libre de Data Leakage, se abrió camino a una optimización bayesiana de alto rendimiento. El modelo final basado en LightGBM no solo alcanza la máxima precisión lograda a lo largo del desarrollo (reduciendo el error a un MAE de $1.97$), sino que exhibe una ligereza computacional apta para integrarse de inmediato en una futura arquitectura de producción.